## Covisitation matrices

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import polars as pl
import polars_ds as pds
import covisits

In [3]:
train = pl.scan_parquet("data/train.parquet")

In [ ]:
%reset -f
#%run gpu-115.ipynb
#%run gpu-116.ipynb
#%run gpu-155.ipynb
#%run gpu-157.ipynb
#%run gpu-165.ipynb
#%run gpu-166.ipynb
%run gpu-167.ipynb
%run gpu-168.ipynb
%run gpu-217.ipynb
%run gpu-220.ipynb
%run gpu-226.ipynb
%run gpu-232.ipynb
%run gpu-235.ipynb
%run gpu-239.ipynb
%run gpu-700.ipynb
%run gpu-701.ipynb
#%run gpu-93.ipynb
#%run cpu-90.ipynb
#%run cpu-95.ipynb
#%run cpu-99.ipynb

### CPU-90

Timings of `make_covisits_90` on full `data/train.parquet` (RTX 2050 4GB, 2026-11-04). All runs give 116,908,684 rows; GPU and streaming results are identical.

| Sessions per chunk | GPU | Streaming |
|---|---|---|
| 250,000 | 54.2s | 28.0s |
| 500,000 | 37.2s | **26.3s** |
| 1,000,000 | 31.7s | 27.7s |
| 2,000,000 | 24.6s | 28.4s |
| 4,000,000 | 22.4s | 29.2s |
| no chunking | **21.4s** | 32.1s |

GPU gets faster with larger chunks; streaming is roughly flat. Defaults are now `engine="gpu"`, `session_chunk=None`. Single timing per setting, so ~1–2s differences may be noise.

#### A100 timings: `make_covisits_90`

**Machine:** RunPod A100-SXM4-80GB, 251 GB RAM, CPU limited by cgroup quota to 13.6 cores (128 visible; RunPod lists 64; Polars uses 13 threads). 2026-11-05, `/opt/.venvs/ML` (polars 1.44.2, cudf-polars-cu13 26.10), `RAPIDSMPF_PINNED_MEMORY=false` (needed: the GPU is on NUMA node 1 and RapidsMPF fails with "No GPUs found on current NUMA node 0"). GPU runs use `pl.GPUEngine(raise_on_fail=True)`, so none fell back to CPU. One process per run, timed from parquet scan to the final top-k, GPU engine start-up excluded. Single timing per setting.

Full `data/train.parquet`, `top=40` (28,492,373 rows). GPU and streaming outputs identical.

| Sessions per chunk | GPU | Streaming |
|---|---|---|
| 250,000 | 17.9s | **18.7s** |
| 500,000 | 15.1s | 19.4s |
| 1,000,000 | 13.4s | 21.1s |
| 2,000,000 | 12.1s | 19.0s |
| 4,000,000 | 11.5s | 19.6s |
| no chunking | **10.8s** | 20.1s |

Same pattern as on the RTX 2050: GPU is fastest unchunked, streaming is flat. The GPU is ~2x faster than on the RTX 2050, streaming ~1.4x.


### A100 timings: all matrices (`save_covisits.py` order)

**Machine:** RunPod A100-SXM4-80GB, 251 GB RAM, CPU limited by cgroup quota to 13.6 cores (128 visible; RunPod lists 64; Polars uses 13 threads). 2026-11-05, `/opt/.venvs/ML` (polars 1.44.2, cudf-polars-cu13 26.10), `RAPIDSMPF_PINNED_MEMORY=false` (needed: the GPU is on NUMA node 1 and RapidsMPF fails with "No GPUs found on current NUMA node 0"). GPU runs use `pl.GPUEngine(raise_on_fail=True)`, so none fell back to CPU. One process per run, timed from parquet scan to the final top-k, GPU engine start-up excluded. Single timing per setting.

Data as in `save_covisits.py`: train + test events (`data/train.parquet` + `data/test.parquet`, 223.6M events), except 155/157 (test only) and 90 (train only, as in the table above). Default `top` per matrix. **GPU** is the fastest setting that ran; chunk = sessions per chunk.

| Matrix | Rows | GPU | GPU setting | Streaming (no chunking) | Speed-up |
|---|---|---|---|---|---|
| 90 | 28,492,373 | 10.8s | no chunking | 20.1s | 1.9x |
| 95 | 68,667,092 | 85.7s | no chunking | 159.0s | 1.9x |
| 99 | 24,920,206 | 8.1s | no chunking | 11.4s | 1.4x |
| 93 | 103,281,791 | 48.7s | chunk 2M (4M 49.7s, 1M 57.7s) | 188.4s | 3.9x |
| 115 | 67,168,397 | 208.3s | chunk 4M (2M 212.8s, 1M 222.6s) | 327.9s | 1.6x |
| 116 | 69,192,950 | 256.0s | chunk 200k (100k 268.9s) | killed, host OOM (chunk 2M 707.7s, 1M 623.7s) | 2.4x vs 1M |
| 155 | 2,969,701 | 0.7s | no chunking | 0.8s | 1.1x |
| 157 | 24,185,781 | 4.2s | no chunking | 5.6s | 1.3x |
| 165 | 1,226,652 | 1.0s | in-memory executor | 5.6s | 5.6x |
| 166 | 60,930,199 | 61.8s | in-memory executor | 111.9s | 1.8x |
| 167 | 44,757,444 | 26.3s | no chunking | 42.3s | 1.6x |
| 168 | 53,683,361 | 40.3s | no chunking | 67.3s | 1.7x |
| 217 | 42,474,738 | 25.4s | chunk 250k | 47.6s | 1.9x |
| 220 | 81,165,344 | 36.1s | chunk 250k | 67.4s | 1.9x |
| 226 | 14,883,417 | 2.5s | no chunking | 4.8s | 1.9x |
| 232 | 94,819,911 | 58.2s | chunk 250k | 183.2s | 3.1x |
| 235 | 117,174,985 | 93.9s | chunk 250k | 285.3s | 3.0x |
| 239 | 21,325,150 | 18.5s | chunk 2M (4M 19.0s, 1M 19.5s) | 27.8s | 1.5x |
| 700 | 2,031,806 | 0.9s | no chunking | 3.4s | 3.8x |
| 701 | 9,256,257 | 2.5s | no chunking | 9.2s | 3.7x |
| **total** | | **~990s (16.5 min)** | | **~2190s (36.5 min)**, 116 at chunk 1M | 2.2x |

**Correctness:** GPU and streaming outputs are identical, except 95, 166, 168 and 232 where 1–3 rows differ at the top-k boundary (float sums differ by < 1e-10, swapping tied rows).

**GPU failures without chunking:**
- 93, 116, 217, 220, 232, 235, 239: out of GPU memory (`MemoryError ... std::bad_alloc`), even on 80 GB.
- 115: `OverflowError` (join output beyond cudf's 2^31-row column limit).
- 116, 217, 220, 232, 235 still fail at 1M–4M sessions per chunk with `OverflowError: The device_uvector size exceeds the column size limit`, and 217–235 run out of memory at 500k. Their session self-joins are huge: summed over sessions, events x unique aids is 10.2B and events^2 is 18.3B; the largest 1M-session chunk has 3.1B (n x u) / 5.7B (n^2). This limit doesn't depend on the GPU, so these chunk sizes also apply to smaller cards. Chris's files had ~130k sessions each.
- 165, 166: `AssertionError: Scan node must have a partition plan` from the default (streaming) GPU executor, on a concat of two parquet scans (`train_test_events`). The in-memory executor (`pl.GPUEngine(executor="in-memory")`) works. Looks like a cudf-polars bug.


In [5]:
%time covisits_90 = covisits.make_covisits_90(train, top=40)

2026-11-04 20:32:11.205 | INFO     | covisits:build_covisits:18 - aggregating chunks
/home/sean/projects/kaggle-otto-multi-objective/.venv/lib/python3.12/site-packages/cudf_polars/streaming/utils.py:97: UserWarning: over(...) inside filter is not supported for multiple partitions; falling back to in-memory evaluation.
  _fallback_inform(msg, rec.state["config_options"])


CPU times: user 23.4 s, sys: 8.28 s, total: 31.7 s
Wall time: 21.4 s


  + Exception Group Traceback (most recent call last):
  |   File "/home/sean/projects/kaggle-otto-multi-objective/.venv/lib/python3.12/site-packages/cudf_polars/engine/core.py", line 672, in execute_ir_on_rank
  |     run_actor_network(ctx, actors=nodes)
  |   File "rapidsmpf/streaming/core/actor.pyx", line 360, in rapidsmpf.streaming.core.actor.run_actor_network
  |   File "rapidsmpf/streaming/core/actor.pyx", line 368, in rapidsmpf.streaming.core.actor.run_actor_network
  |   File "rapidsmpf/streaming/core/actor.pyx", line 365, in rapidsmpf.streaming.core.actor.run_actor_network
  |   File "/home/sean/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/concurrent/futures/_base.py", line 456, in result
  |     return self.__get_result()
  |            ^^^^^^^^^^^^^^^^^^^
  |   File "/home/sean/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/concurrent/futures/_base.py", line 401, in __get_result
  |     raise self._exception
  |   File "/home

In [8]:
covisits_90.head()

aid,aid_y,value
i32,i32,u32
1,886584,1
1,279811,1
1,1708199,1
1,850666,1
1,1832177,1


In [12]:
zz = covisits.top_covisit(covisits_90)

In [18]:
zz.shape

(28492373, 3)

In [16]:
covisits_90.describe()

statistic,aid,aid_y,value
str,f64,f64,f64
"""count""",1.16908684e8,1.16908684e8,1.16908684e8
"""null_count""",0.0,0.0,0.0
"""mean""",929452.681479,929452.681479,1.142985
"""std""",535367.06466,535367.06466,1.660367
"""min""",1.0,1.0,1.0
"""25%""",467265.0,467265.0,1.0
"""50%""",930835.0,930835.0,1.0
"""75%""",1.394286e6,1.394286e6,1.0
"""max""",1.855601e6,1.855601e6,2465.0


In [14]:
zz.describe()

statistic,aid,aid_y,value
str,f64,f64,f64
"""count""",2.8492373e7,2.8492373e7,2.8492373e7
"""null_count""",0.0,0.0,0.0
"""mean""",928603.411956,929359.983967,1.373906
"""std""",535583.017249,535710.021038,3.228221
"""min""",1.0,1.0,1.0
"""25%""",465202.0,466271.0,1.0
"""50%""",928517.0,931139.0,1.0
"""75%""",1.392504e6,1.394717e6,1.0
"""max""",1.855601e6,1.855601e6,2465.0
